# Machine Learning Models

### Imports and Set up

In [24]:
import numpy as np
import pandas as pd
import pickle
import time
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

### Load data

In [ ]:
with open('group_5_dataset.pkl', 'rb') as f:
    data = pickle.load(f)

# data format from pickle file:
# 'X_train': (n_train, T, 4),
# 'X_val':   (n_val,   T, 4),
# 'X_test':  (n_test,  T, 4),
# 'y_train': (n_train, 2),   # [R, C]
# 'y_val':   (n_val,   2),
# 'y_test':  (n_test,  2),
# optional: 'mu', 'sigma', 'indices', 'target_is_log'
target_is_log = data.get('target_is_log', False)

# Keep time-series waveforms and scaling stats for Gauss-Newton
X_test_ts = data['X_test']                   # (n_test, T, 4)
mu    = data.get('mu',    None)               # (1, 1, 4) channel means
sigma = data.get('sigma', None)               # (1, 1, 4) channel stds

# flatten out the data from pickle file
X_train = data['X_train'].reshape(data['X_train'].shape[0], -1)  # (n_train, 2000)
X_val = data['X_val'].reshape(data['X_val'].shape[0], -1)        # (n_val, 2000)
X_test = data['X_test'].reshape(data['X_test'].shape[0], -1)     # (n_test, 2000)
y_train = data['y_train']  # (n_train, 2)
y_val = data['y_val']       # (n_val, 2)
y_test = data['y_test']    # (n_test, 2)

### Data preview 

Each sample has **2000 features**: 500 time steps × 4 channels (V1, V2, V3, I_E). At each time step the 4 values are [V1, V2, V3, I_E]. Features 0–3 = first time step, 4–7 = second, etc. Below: first 8 (2 time steps) + targets R (Ω), C (F).

In [26]:
# First 20 rows: first 8 features = first 2 time steps (each step: V1, V2, V3, I_E) + targets
n_display = 20
n_feat = X_train.shape[1]
channel_names = ["V1", "V2", "V3", "I_E"]
n_show = min(8, n_feat)
cols = [f"t{i//4}_{channel_names[i%4]}" for i in range(n_show)]
df_feat = pd.DataFrame(X_train[:n_display, :n_show], columns=cols)
df_targets = pd.DataFrame(y_train[:n_display], columns=["R (ohm)", "C (F)"])
df_preview = pd.concat([df_feat, df_targets], axis=1)
print("First 20 samples (first 2 time steps: t0_V1..t0_I_E, t1_V1..t1_I_E) + targets:")
display(df_preview)
print(f"\nTotal features per sample: {X_train.shape[1]} (500 steps × 4 channels). Target ranges — R: [{y_train[:, 0].min():.2f}, {y_train[:, 0].max():.2f}] Ω,  C: [{y_train[:, 1].min():.2e}, {y_train[:, 1].max():.2e}] F")

First 20 samples (first 2 time steps: t0_V1..t0_I_E, t1_V1..t1_I_E) + targets:


,t0_V1,t0_V2,t0_V3,t0_I_E,t1_V1,t1_V2,t1_V3,t1_I_E,R (ohm),C (F)
0,0.016586,0.048113,-1.152401,0.437004,0.084029,0.063066,-1.111389,0.418302,337.941720,2.581128e-06
1,-0.015418,0.008174,-1.134681,0.417119,0.082737,0.056419,-1.161826,0.442741,1043.441926,2.418469e-06
2,-0.001115,0.059448,-1.128545,0.430000,0.078711,0.121899,-1.109472,0.423079,2363.432484,1.546963e-06
3,-0.006379,0.042218,-1.148456,0.428323,0.081132,0.112252,-1.102958,0.420738,1288.372037,1.228863e-07
4,-0.009116,0.008465,-1.095387,0.399199,0.070763,0.100985,-1.095129,0.424032,265.667313,1.844064e-06
5,-0.018989,0.023984,-1.108714,0.425062,0.024625,0.130615,-1.148364,0.423264,1961.738470,4.524646e-07
6,0.010389,0.036250,-1.118222,0.430049,0.002981,0.136076,-1.116253,0.428996,2352.190259,4.457801e-06
7,0.000031,0.045366,-1.145943,0.428431,0.039668,0.083389,-1.139877,0.437478,2030.765555,4.879987e-06
8,-0.035834,0.007595,-1.143327,0.401294,0.007217,0.107501,-1.087426,0.398724,732.387275,3.794111e-06
9,0.023900,0.054853,-1.144349,0.408578,0.029403,0.063983,-1.103123,0.410224,1394.308616,4.374113e-06



Total features per sample: 2000 (500 steps × 4 channels). Target ranges — R: [2.66, 2499.18] Ω,  C: [1.02e-07, 5.00e-06] F


### Train / validation / test split

In [27]:
n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_total = n_train + n_val + n_test
split_df = pd.DataFrame({
    "set": ["train", "validation", "test"],
    "samples": [n_train, n_val, n_test],
    "fraction (%)": [100 * n_train / n_total, 100 * n_val / n_total, 100 * n_test / n_total],
})
print("Split (from pickle):")
display(split_df)
print(f"Total samples: {n_total}  |  Features per sample: {X_train.shape[1]}  |  Targets: R, C")

Split (from pickle):


,set,samples,fraction (%)
0,train,1400,70.0
1,validation,300,15.0
2,test,300,15.0


Total samples: 2000  |  Features per sample: 2000  |  Targets: R, C


## 1. Linear regression

**Hyperparameter tuning:** We use **Ridge regression** (linear model with L2 regularization) and tune the regularization strength `alpha` on the validation set. For each configuration we record train, validation, and test MAE and RMSE (for R and C), training time, and data quantity. The best configuration is chosen by validation accuracy (lowest average validation MAE across R and C).

In [28]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys = np.exp(y_val) if target_is_log else y_val
y_test_phys = np.exp(y_test) if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def metrics_r_c(y_true, y_pred):
    mae_r = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    return mae_r, mae_c, rmse_r, rmse_c

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
alpha_list = [1e-4, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0]
lr_results = []
best_val_mae_avg = float('inf')
lr_model = None
train_time_lr = None
best_lr_params = {}

for alpha in alpha_list:
    model = Ridge(alpha=alpha, random_state=42)
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    t_elapsed = time.perf_counter() - t0
    y_train_pred = to_phys(model.predict(X_train), target_is_log)
    y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
    y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
    tr = metrics_r_c(y_train_phys, y_train_pred)
    va = metrics_r_c(y_val_phys,   y_val_pred)
    te = metrics_r_c(y_test_phys,  y_test_pred)
    val_mae_avg = (va[0] + va[1]) / 2
    lr_results.append({
        'alpha': alpha,
        'train_time_s': t_elapsed,
        'train_mae_r': tr[0], 'train_mae_c': tr[1],
        'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
        'val_mae_r': va[0], 'val_mae_c': va[1],
        'val_rmse_r': va[2], 'val_rmse_c': va[3],
        'test_mae_r': te[0], 'test_mae_c': te[1],
        'test_rmse_r': te[2], 'test_rmse_c': te[3],
    })
    if val_mae_avg < best_val_mae_avg:
        best_val_mae_avg = val_mae_avg
        lr_model = model
        train_time_lr = t_elapsed
        best_lr_params = {'alpha': alpha}

lr_experiments_df = pd.DataFrame(lr_results)
print("Linear Regression (Ridge) — hyperparameter tuning experiments")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Note: best config selected by validation MAE only; test metrics shown for reference.")
c_cols_lr = [c for c in lr_experiments_df.columns if '_mae_c' in c or '_rmse_c' in c]
r_cols_lr  = [c for c in lr_experiments_df.columns if ('_mae_r' in c or '_rmse_r' in c)]
t_cols_lr  = ['train_time_s']
fmt_lr = {col: "{:.4e}" for col in c_cols_lr}
fmt_lr.update({col: "{:.4f}" for col in r_cols_lr})
fmt_lr.update({col: "{:.4f}" for col in t_cols_lr})
display(lr_experiments_df.style.format(fmt_lr))
print(f"\nBest config (by validation MAE): {best_lr_params}")
print("Justification: Chosen alpha minimizes average validation MAE (R and C).")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(lr_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(lr_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_lr:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}")


Linear Regression (Ridge) — hyperparameter tuning experiments
Data: n_train=1400, n_val=300, n_test=300
Note: best config selected by validation MAE only; test metrics shown for reference.


,alpha,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c
0,0.000100,0.0983,0.0619,3.2523e-11,0.0801,4.0899e-11,170.7563,1.0415e-07,230.1935,1.4722e-07,189.5072,1.1006e-07,243.4628,1.9785e-07
1,0.001000,0.0951,0.6066,3.1901e-10,0.7851,4.0121e-10,169.3661,1.0346e-07,228.4090,1.4631e-07,187.9271,1.0932e-07,241.5494,1.9717e-07
2,0.010000,0.0937,5.0994,2.7093e-09,6.6200,3.4113e-09,158.2299,9.7830e-08,214.3783,1.3933e-07,175.3510,1.0353e-07,226.7856,1.9185e-07
3,0.100000,0.0894,23.0599,1.2940e-08,30.7740,1.6729e-08,123.0199,7.8145e-08,171.1676,1.2058e-07,136.6843,8.4971e-08,183.3063,1.7606e-07
4,1.000000,0.0935,52.8352,3.1154e-08,75.8085,4.8723e-08,97.9931,6.6875e-08,152.3405,1.2093e-07,102.3611,6.9033e-08,154.6252,1.6483e-07
5,10.000000,0.0953,107.6507,5.6375e-08,152.4324,1.1985e-07,127.3799,7.1663e-08,200.7950,1.5035e-07,127.5018,7.2383e-08,192.8623,1.7610e-07
6,100.000000,0.0966,158.3843,9.8728e-08,219.6415,1.8414e-07,170.3916,1.0836e-07,255.7824,1.8679e-07,175.3198,1.0657e-07,249.3188,2.0712e-07



Best config (by validation MAE): {'alpha': 1.0}
Justification: Chosen alpha minimizes average validation MAE (R and C).

Best model — Val and Test metrics:
  Train time: 0.0935 s
  Val:  R mae: 97.9931  C mae: 6.6875e-08  R rmse: 152.3405  C rmse: 1.2093e-07
  Test: R mae: 102.3611  C mae: 6.9033e-08  R rmse: 154.6252  C rmse: 1.6483e-07


## 2. Random forest

**Hyperparameter tuning:** We tune `n_estimators` and `max_depth` on the validation set. For each configuration we record train, validation, and test MAE and RMSE (for R and C), training time, and data quantity. The best configuration is selected by validation accuracy (lowest average validation MAE across R and C).

In [30]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys = np.exp(y_val) if target_is_log else y_val
y_test_phys = np.exp(y_test) if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def metrics_r_c(y_true, y_pred):
    mae_r = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    return mae_r, mae_c, rmse_r, rmse_c

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
n_estimators_list = [10, 50, 100, 200]
max_depth_list = [5, 10, 20, None]
rf_results = []
best_model = None
best_val_mae_avg = float('inf')
best_rf_params = {}
train_time_rf = None

for n_estimators in n_estimators_list:
    for max_depth in max_depth_list:
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        t_elapsed = time.perf_counter() - t0
        y_train_pred = to_phys(model.predict(X_train), target_is_log)
        y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
        y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
        tr = metrics_r_c(y_train_phys, y_train_pred)
        va = metrics_r_c(y_val_phys,   y_val_pred)
        te = metrics_r_c(y_test_phys,  y_test_pred)
        val_mae_avg = (va[0] + va[1]) / 2
        rf_results.append({
            'n_estimators': n_estimators,
            'max_depth': str(max_depth),
            'train_time_s': t_elapsed,
            'train_mae_r': tr[0], 'train_mae_c': tr[1],
            'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
            'val_mae_r': va[0], 'val_mae_c': va[1],
            'val_rmse_r': va[2], 'val_rmse_c': va[3],
            'test_mae_r': te[0], 'test_mae_c': te[1],
            'test_rmse_r': te[2], 'test_rmse_c': te[3],
        })
        if val_mae_avg < best_val_mae_avg:
            best_val_mae_avg = val_mae_avg
            best_model = model
            best_rf_params = {'n_estimators': n_estimators, 'max_depth': max_depth}
            train_time_rf = t_elapsed

rf_experiments_df = pd.DataFrame(rf_results)
print("Random Forest — hyperparameter tuning experiments")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Note: best config selected by validation MAE only; test metrics shown for reference.")
c_cols_rf = [c for c in rf_experiments_df.columns if '_mae_c' in c or '_rmse_c' in c]
r_cols_rf  = [c for c in rf_experiments_df.columns if ('_mae_r' in c or '_rmse_r' in c)]
t_cols_rf  = ['train_time_s']
fmt_rf = {col: "{:.4e}" for col in c_cols_rf}
fmt_rf.update({col: "{:.4f}" for col in r_cols_rf})
fmt_rf.update({col: "{:.4f}" for col in t_cols_rf})
display(rf_experiments_df.style.format(fmt_rf))
print(f"\nBest config (by validation MAE): {best_rf_params}")
print("Justification: Chosen n_estimators and max_depth minimize average validation MAE (R and C).")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(best_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(best_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_rf:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}")


Random Forest — hyperparameter tuning experiments
Data: n_train=1400, n_val=300, n_test=300
Note: best config selected by validation MAE only; test metrics shown for reference.


,n_estimators,max_depth,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c
0,10,5,10.6317,20.3360,1.1720e-06,27.9431,1.3573e-06,28.3269,1.1310e-06,41.3911,1.3444e-06,26.2766,1.1564e-06,36.2746,1.3485e-06
1,10,10,17.2077,8.4482,5.3076e-07,13.0168,6.6052e-07,21.4869,8.2044e-07,34.9756,1.0055e-06,19.9978,7.9799e-07,30.8450,9.7077e-07
2,10,20,18.5582,8.4110,3.0245e-07,12.8902,4.1549e-07,21.8823,7.4714e-07,35.2363,9.5123e-07,20.1927,7.2146e-07,30.1850,8.9550e-07
3,10,None,19.2688,8.4110,3.0245e-07,12.8902,4.1549e-07,21.8823,7.4714e-07,35.2363,9.5123e-07,20.1927,7.2146e-07,30.1850,8.9550e-07
4,50,5,50.5317,18.5064,1.1620e-06,25.8488,1.3451e-06,26.6345,1.1243e-06,39.9122,1.3344e-06,24.6898,1.1519e-06,34.5302,1.3399e-06
5,50,10,86.0953,6.8939,5.0321e-07,10.3628,6.0154e-07,19.2079,7.9572e-07,32.0624,9.5501e-07,18.2129,7.9867e-07,27.7454,9.4559e-07
6,50,20,87.4142,6.8884,2.7336e-07,10.2894,3.3897e-07,19.6231,7.0558e-07,32.5376,8.6535e-07,18.4942,7.1691e-07,28.0823,8.5464e-07
7,50,None,87.6513,6.8884,2.7336e-07,10.2894,3.3897e-07,19.6231,7.0558e-07,32.5376,8.6535e-07,18.4942,7.1691e-07,28.0823,8.5464e-07
8,100,5,102.4830,18.3843,1.1555e-06,25.7111,1.3382e-06,26.6204,1.1178e-06,40.2830,1.3256e-06,24.3236,1.1446e-06,34.0539,1.3321e-06
9,100,10,173.8846,6.6533,4.9843e-07,10.0113,5.9367e-07,19.2361,7.9037e-07,32.1945,9.4768e-07,17.8955,7.9533e-07,27.0523,9.3651e-07



Best config (by validation MAE): {'n_estimators': 200, 'max_depth': 10}
Justification: Chosen n_estimators and max_depth minimize average validation MAE (R and C).

Best model — Val and Test metrics:
  Train time: 424.5419 s
  Val:  R mae: 18.7660  C mae: 7.8783e-07  R rmse: 31.2355  C rmse: 9.4492e-07
  Test: R mae: 17.5957  C mae: 7.9298e-07  R rmse: 26.4364  C rmse: 9.3445e-07


## 3. MLP

**Hyperparameter tuning:** We tune `hidden_layer_sizes` and L2 regularization `alpha` on the validation set. For each configuration we record train, validation, and test MAE and RMSE (for R and C), training time, and data quantity. The best configuration is selected by validation accuracy (lowest average validation MAE across R and C).

In [29]:
target_is_log = globals().get('target_is_log', False)
y_train_phys = np.exp(y_train) if target_is_log else y_train
y_val_phys = np.exp(y_val) if target_is_log else y_val
y_test_phys = np.exp(y_test) if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def metrics_r_c(y_true, y_pred):
    mae_r = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    return mae_r, mae_c, rmse_r, rmse_c

n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
hidden_sizes_list = [(64, 32), (128, 64), (256, 128, 64)]
alpha_list = [1e-4, 1e-3, 1e-2]
mlp_results = []
mlp_model = None
best_val_mae_avg = float('inf')
best_mlp_params = {}
train_time_mlp = None

for hidden in hidden_sizes_list:
    for alpha in alpha_list:
        model = MLPRegressor(
            hidden_layer_sizes=hidden,
            activation='relu',
            solver='adam',
            alpha=alpha,
            max_iter=500,
            random_state=42,
            early_stopping=True,
            validation_fraction=0.1,
        )
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        t_elapsed = time.perf_counter() - t0
        y_train_pred = to_phys(model.predict(X_train), target_is_log)
        y_val_pred   = to_phys(model.predict(X_val),   target_is_log)
        y_test_pred  = to_phys(model.predict(X_test),  target_is_log)
        tr = metrics_r_c(y_train_phys, y_train_pred)
        va = metrics_r_c(y_val_phys,   y_val_pred)
        te = metrics_r_c(y_test_phys,  y_test_pred)
        val_mae_avg = (va[0] + va[1]) / 2
        mlp_results.append({
            'hidden_layer_sizes': str(hidden),
            'alpha': alpha,
            'train_time_s': t_elapsed,
            'train_mae_r': tr[0], 'train_mae_c': tr[1],
            'train_rmse_r': tr[2], 'train_rmse_c': tr[3],
            'val_mae_r': va[0], 'val_mae_c': va[1],
            'val_rmse_r': va[2], 'val_rmse_c': va[3],
            'test_mae_r': te[0], 'test_mae_c': te[1],
            'test_rmse_r': te[2], 'test_rmse_c': te[3],
        })
        if val_mae_avg < best_val_mae_avg:
            best_val_mae_avg = val_mae_avg
            mlp_model = model
            best_mlp_params = {'hidden_layer_sizes': hidden, 'alpha': alpha}
            train_time_mlp = t_elapsed

mlp_experiments_df = pd.DataFrame(mlp_results)
print("MLP — hyperparameter tuning experiments")
print(f"Data: n_train={n_train}, n_val={n_val}, n_test={n_test}")
print("Note: best config selected by validation MAE only; test metrics shown for reference.")
c_cols_mlp = [c for c in mlp_experiments_df.columns if '_mae_c' in c or '_rmse_c' in c]
r_cols_mlp  = [c for c in mlp_experiments_df.columns if ('_mae_r' in c or '_rmse_r' in c)]
t_cols_mlp  = ['train_time_s']
fmt_mlp = {col: "{:.4e}" for col in c_cols_mlp}
fmt_mlp.update({col: "{:.4f}" for col in r_cols_mlp})
fmt_mlp.update({col: "{:.4f}" for col in t_cols_mlp})
display(mlp_experiments_df.style.format(fmt_mlp))
print(f"\nBest config (by validation MAE): {best_mlp_params}")
print("Justification: Chosen hidden_layer_sizes and alpha minimize average validation MAE (R and C).")
print("\nBest model — Val and Test metrics:")
y_val_pred  = to_phys(mlp_model.predict(X_val),  target_is_log)
y_test_pred = to_phys(mlp_model.predict(X_test), target_is_log)
va = metrics_r_c(y_val_phys,  y_val_pred)
te = metrics_r_c(y_test_phys, y_test_pred)
print(f"  Train time: {train_time_mlp:.4f} s")
print(f"  Val:  R mae: {va[0]:.4f}  C mae: {va[1]:.4e}  R rmse: {va[2]:.4f}  C rmse: {va[3]:.4e}")
print(f"  Test: R mae: {te[0]:.4f}  C mae: {te[1]:.4e}  R rmse: {te[2]:.4f}  C rmse: {te[3]:.4e}")


MLP — hyperparameter tuning experiments
Data: n_train=1400, n_val=300, n_test=300
Note: best config selected by validation MAE only; test metrics shown for reference.


,hidden_layer_sizes,alpha,train_time_s,train_mae_r,train_mae_c,train_rmse_r,train_rmse_c,val_mae_r,val_mae_c,val_rmse_r,val_rmse_c,test_mae_r,test_mae_c,test_rmse_r,test_rmse_c
0,"(64, 32)",0.000100,1.1127,487.3174,4.2260e+00,560.9775,6.7118e+00,522.6256,4.4033e+00,596.8071,6.8852e+00,473.6173,4.4228e+00,555.4579,6.9337e+00
1,"(64, 32)",0.001000,2.2025,224.0640,1.4715e+00,287.2421,2.3036e+00,237.5641,1.5871e+00,316.3415,2.4817e+00,234.9977,1.4702e+00,301.7333,2.4388e+00
2,"(64, 32)",0.010000,1.4881,546.0549,8.0745e+00,624.3131,1.2478e+01,584.4564,8.5047e+00,660.8945,1.3182e+01,533.8519,8.4638e+00,618.9535,1.2963e+01
3,"(128, 64)",0.000100,1.5591,1218.8151,1.1546e+00,1412.5321,1.2034e+00,1245.2700,1.1752e+00,1458.4012,1.2499e+00,1209.1414,1.1557e+00,1404.6091,1.2172e+00
4,"(128, 64)",0.001000,1.3029,1218.8182,1.1783e+00,1412.5351,1.2260e+00,1245.2729,1.1989e+00,1458.4044,1.2708e+00,1209.1447,1.1800e+00,1404.6126,1.2394e+00
5,"(128, 64)",0.010000,1.4721,1218.8085,1.1981e+00,1412.5267,1.2450e+00,1245.2632,1.2184e+00,1458.3960,1.2882e+00,1209.1350,1.2000e+00,1404.6042,1.2579e+00
6,"(256, 128, 64)",0.000100,4.0794,715.3031,1.6143e+00,865.9594,1.6809e+00,763.8052,1.6518e+00,919.5138,1.7521e+00,712.2641,1.6316e+00,859.2161,1.7068e+00
7,"(256, 128, 64)",0.001000,3.0423,714.7842,1.6399e+00,865.2595,1.7142e+00,763.3183,1.6803e+00,918.8096,1.7933e+00,711.7449,1.6604e+00,858.5168,1.7465e+00
8,"(256, 128, 64)",0.010000,4.2721,714.1404,1.6950e+00,864.4578,1.7749e+00,762.7029,1.7395e+00,918.0021,1.8650e+00,711.1012,1.7133e+00,857.7181,1.8071e+00



Best config (by validation MAE): {'hidden_layer_sizes': (64, 32), 'alpha': 0.001}
Justification: Chosen hidden_layer_sizes and alpha minimize average validation MAE (R and C).

Best model — Val and Test metrics:
  Train time: 2.2025 s
  Val:  R mae: 237.5641  C mae: 1.5871e+00  R rmse: 316.3415  C rmse: 2.4817e+00
  Test: R mae: 234.9977  C mae: 1.4702e+00  R rmse: 301.7333  C rmse: 2.4388e+00


## Model comparison

The table below compares all three models on the **test set** using MAE and RMSE for R (Ω) and C (F), plus total time (training + test inference). The best hyperparameters were selected using the **validation set only**.

In [31]:
target_is_log = globals().get('target_is_log', False)
y_test_phys = np.exp(y_test) if target_is_log else y_test

def to_phys(y_pred, is_log):
    return np.exp(y_pred) if is_log else y_pred

def metrics_r_c(y_true, y_pred):
    mae_r = mean_absolute_error(y_true[:, 0], y_pred[:, 0])
    mae_c = mean_absolute_error(y_true[:, 1], y_pred[:, 1])
    rmse_r = np.sqrt(mean_squared_error(y_true[:, 0], y_pred[:, 0]))
    rmse_c = np.sqrt(mean_squared_error(y_true[:, 1], y_pred[:, 1]))
    return mae_r, mae_c, rmse_r, rmse_c

# Measure test-set inference time for each best model
t0 = time.perf_counter(); lr_test_pred  = to_phys(lr_model.predict(X_test),    target_is_log); infer_time_lr  = time.perf_counter() - t0
t0 = time.perf_counter(); rf_test_pred  = to_phys(best_model.predict(X_test),  target_is_log); infer_time_rf  = time.perf_counter() - t0
t0 = time.perf_counter(); mlp_test_pred = to_phys(mlp_model.predict(X_test),   target_is_log); infer_time_mlp = time.perf_counter() - t0

# Compute test metrics
lr_te  = metrics_r_c(y_test_phys, lr_test_pred)
rf_te  = metrics_r_c(y_test_phys, rf_test_pred)
mlp_te = metrics_r_c(y_test_phys, mlp_test_pred)

# Total time = train + inference
total_time_lr  = train_time_lr  + infer_time_lr
total_time_rf  = train_time_rf  + infer_time_rf
total_time_mlp = train_time_mlp + infer_time_mlp

rows = [
    {"Method": "Linear Regression (Ridge)",
     "MAE (R)": lr_te[0],  "MAE (C)": lr_te[1],
     "RMSE (R)": lr_te[2], "RMSE (C)": lr_te[3],
     "Time (s)": total_time_lr},
    {"Method": "Random Forest",
     "MAE (R)": rf_te[0],  "MAE (C)": rf_te[1],
     "RMSE (R)": rf_te[2], "RMSE (C)": rf_te[3],
     "Time (s)": total_time_rf},
    {"Method": "MLP",
     "MAE (R)": mlp_te[0],  "MAE (C)": mlp_te[1],
     "RMSE (R)": mlp_te[2], "RMSE (C)": mlp_te[3],
     "Time (s)": total_time_mlp},
]

comparison_df = pd.DataFrame(rows)
fmt = {
    "MAE (R)":  "{:.4f}",
    "RMSE (R)": "{:.4f}",
    "MAE (C)":  "{:.4e}",
    "RMSE (C)": "{:.4e}",
    "Time (s)": "{:.4f}",
}
display(comparison_df.style.format(fmt).hide(axis="index"))


Method,MAE (R),MAE (C),RMSE (R),RMSE (C),Time (s)
Linear Regression (Ridge),102.3611,6.9033e-08,154.6252,1.6483e-07,0.0986
Random Forest,17.5957,7.9298e-07,26.4364,9.3445e-07,424.5689
MLP,234.9977,1.4702e+00,301.7333,2.4388e+00,2.2088


## Best hyperparameters per model

The table below summarises the best hyperparameter configuration found for each model (chosen by lowest average validation MAE across R and C), along with data quantities and training time.

In [32]:
n_train, n_val, n_test = len(X_train), len(X_val), len(X_test)
data_str = f"({n_train}, {n_val}, {n_test})"

summary_rows = [
    {
        "Model": "Linear Regression (Ridge)",
        "Best hyperparameters": f"alpha={best_lr_params.get('alpha', '—')}",
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_lr,
    },
    {
        "Model": "Random Forest",
        "Best hyperparameters": (
            f"n_estimators={best_rf_params.get('n_estimators', '—')}, "
            f"max_depth={best_rf_params.get('max_depth', '—')}"
        ),
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_rf,
    },
    {
        "Model": "MLP",
        "Best hyperparameters": (
            f"hidden_layer_sizes={best_mlp_params.get('hidden_layer_sizes', '—')}, "
            f"alpha={best_mlp_params.get('alpha', '—')}"
        ),
        "Data (n_train, n_val, n_test)": data_str,
        "Train time (s)": train_time_mlp,
    },
]

summary_df = pd.DataFrame(summary_rows)
display(summary_df.style.format({"Train time (s)": "{:.4f}"}).hide(axis="index"))


Model,Best hyperparameters,"Data (n_train, n_val, n_test)",Train time (s)
Linear Regression (Ridge),alpha=1.0,"(1400, 300, 300)",0.0935
Random Forest,"n_estimators=200, max_depth=10","(1400, 300, 300)",424.5419
MLP,"hidden_layer_sizes=(64, 32), alpha=0.001","(1400, 300, 300)",2.2025


## Hybrid ML + Gauss-Newton

### How the models are combined

The best ML model (Random Forest) is used as a **warm-start initializer** for the Gauss-Newton (GN) parameter estimator from `CircuitSimulator`:

1. **Step 1 - ML prediction:** Random Forest predicts initial estimates of R and C directly from the flattened waveform features. This is instantaneous and requires no knowledge of the circuit equations.
2. **Step 2 - Gauss-Newton refinement:** The predicted (R, C) are passed as the starting point to `CircuitSimulator.GaussNewton`, which minimizes the residual between the circuit-simulated waveform and the actual observed waveform. At each iteration it:
   - Runs a full Backward Euler simulation with the current (R, C) guess.
   - Computes the sensitivity (Jacobian) of each simulated signal with respect to R and C.
   - Updates (R, C) via the multiplicative GN step: beta = (J^T J)^-1 J^T r, then R *= exp(beta[0]), C *= exp(beta[1]).

### Why this combination improves accuracy

| Problem with ML alone | How Gauss-Newton corrects it |
|---|---|
| Predictions are unconstrained - ML may output physically inconsistent (R, C) | GN enforces physics: the final answer must be consistent with the circuit ODE |
| ML generalises from training data; waveform shapes outside the training distribution degrade accuracy | GN minimises waveform residuals directly, so it adapts to each individual sample |
| ML has systematic bias from limited training data | GN iteratively reduces residuals toward zero |
| Gauss-Newton alone is sensitive to initialisation and can diverge | ML provides a close warm start, making GN converge faster and reliably |

### Trade-offs

- **Computation:** Much slower than ML alone - each GN iteration requires a full Backward Euler simulation per sample.
- **Dependencies:** Requires the physics simulator and physical-unit waveforms (the dataset is stored standardized, so unstandardization must be applied before calling GN).
- **Best use case:** High-accuracy post-processing of a small number of predictions where accuracy matters more than throughput.


In [38]:
from circuit_simulator import CircuitSimulator

# Circuit simulation constants (matching dataset generation in test.py)
AMPLITUDE = 5.0
FREQUENCY = 60.0
DELTA_T   = 1e-4
# Use first 100 steps to keep each GN call fast (~0.5 s / sample)
N_STEPS   = 100
# Tiny epsilon avoids an off-by-one due to float rounding in BEuler's while(t < T)
T_SUB     = N_STEPS * DELTA_T - 1e-12

def unstandardize(waveform_std, mu, sigma):
    """Convert standardized (T,4) waveform back to physical units."""
    if mu is None or sigma is None:
        return waveform_std
    return waveform_std * sigma[0] + mu[0]

# If the load-data cell wasn't rerun, pull waveforms/scaling directly from the pickle
try:
    X_test_ts
except NameError:
    import pickle
    with open('group_5_dataset.pkl', 'rb') as f:
        _data_gn = pickle.load(f)
    X_test_ts = _data_gn['X_test']
    mu = _data_gn.get('mu', None)
    sigma = _data_gn.get('sigma', None)

target_is_log = globals().get('target_is_log', False)
sample_indices = [0, 1, 2, 3, 4]

# RF predictions (physical units) for the chosen test samples
rf_init = to_phys(best_model.predict(X_test[sample_indices]), target_is_log)
y_true_phys = np.exp(y_test[sample_indices]) if target_is_log else y_test[sample_indices]

hybrid_rows = []
for j, idx in enumerate(sample_indices):
    R_true, C_true = y_true_phys[j, 0], y_true_phys[j, 1]
    R_ml,   C_ml   = rf_init[j, 0],     rf_init[j, 1]

    # Unstandardize waveform back to physical units, then truncate to N_STEPS
    waveform = unstandardize(X_test_ts[idx], mu, sigma)[:N_STEPS]

    # Run Gauss-Newton starting from ML prediction
    mna = CircuitSimulator(AMPLITUDE, FREQUENCY, R_ml, C_ml)
    R_hybrid, C_hybrid, _ = mna.GaussNewton(
        R_ml, C_ml, np.zeros(4), waveform, DELTA_T, T_SUB, max_iter=10
    )

    hybrid_rows.append({
        "idx":            idx,
        "R_true":         R_true,
        "C_true":         C_true,
        "R_ml":           R_ml,
        "C_ml":           C_ml,
        "R_hybrid":       R_hybrid,
        "C_hybrid":       C_hybrid,
        "|err_R| ML":     abs(R_ml     - R_true),
        "|err_R| Hybrid": abs(R_hybrid - R_true),
        "|err_C| ML":     abs(C_ml     - C_true),
        "|err_C| Hybrid": abs(C_hybrid - C_true),
    })

hybrid_df = pd.DataFrame(hybrid_rows).set_index("idx")
fmt_h = {
    "R_true": "{:.2f}", "R_ml": "{:.2f}", "R_hybrid": "{:.2f}",
    "|err_R| ML": "{:.4f}", "|err_R| Hybrid": "{:.4f}",
    "C_true": "{:.4e}", "C_ml": "{:.4e}", "C_hybrid": "{:.4e}",
    "|err_C| ML": "{:.4e}", "|err_C| Hybrid": "{:.4e}",
}
print(f"Hybrid ML + Gauss-Newton: parameter estimation on {len(sample_indices)} test samples")
print(f"(GN uses first {N_STEPS} time steps, max_iter=10, warm-started from Random Forest)")
display(hybrid_df.style.format(fmt_h))

mae_r_ml     = hybrid_df["|err_R| ML"].mean()
mae_r_hybrid = hybrid_df["|err_R| Hybrid"].mean()
mae_c_ml     = hybrid_df["|err_C| ML"].mean()
mae_c_hybrid = hybrid_df["|err_C| Hybrid"].mean()
print(f"\nMean |err R|: ML = {mae_r_ml:.4f} ohm  -->  Hybrid = {mae_r_hybrid:.4f} ohm  "
      f"({'improved' if mae_r_hybrid < mae_r_ml else 'no improvement'})")
print(f"Mean |err C|: ML = {mae_c_ml:.4e} F    -->  Hybrid = {mae_c_hybrid:.4e} F    "
      f"({'improved' if mae_c_hybrid < mae_c_ml else 'no improvement'})")


Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Number of iterations: 10.
Hybrid ML + Gauss-Newton: parameter estimation on 5 test samples
(GN uses first 100 time steps, max_iter=10, warm-started from Random Forest)


,R_true,C_true,R_ml,C_ml,R_hybrid,C_hybrid,|err_R| ML,|err_R| Hybrid,|err_C| ML,|err_C| Hybrid
idx,,,,,,,,,,
0,1088.91,3.2798e-06,1112.38,2.6357e-06,1140.88,3.1124e-06,23.4657,51.9696,6.4411e-07,1.6743e-07
1,501.39,1.5867e-06,508.79,1.9506e-06,492.76,1.5803e-06,7.3993,8.6257,3.6392e-07,6.4441e-09
2,2426.10,1.4454e-06,2441.97,2.0052e-06,2493.38,1.4007e-06,15.8725,67.2858,5.5982e-07,4.4708e-08
3,1861.51,2.0618e-06,1865.98,2.3862e-06,1891.87,2.0143e-06,4.4643,30.3541,3.2438e-07,4.7501e-08
4,708.21,3.8566e-06,715.29,2.9227e-06,716.60,3.8045e-06,7.0776,8.3903,9.3390e-07,5.2119e-08



Mean |err R|: ML = 11.6559 ohm  -->  Hybrid = 33.3251 ohm  (no improvement)
Mean |err C|: ML = 5.6522e-07 F    -->  Hybrid = 6.3641e-08 F    (improved)
